# Waterfall Plot

A waterfall plot shows the cumulative effect of sequentially introduced positive or negative values.

In [1]:
%useLatestDescriptors
%use lets-plot

In [2]:
val dataMap = mapOf(
    "Accounts" to listOf("Product revenue", "Services revenue", "Fixed costs", "Variable costs"),
    "Values" to listOf(830_000, 290_000, -360_000, -150_000),
)

## Default View

In [3]:
waterfallPlot(dataMap, "Accounts", "Values")

## Improved View

In [4]:
waterfallPlot(dataMap, "Accounts", "Values",
              size = 0.75, width = 0.8, totalTitle = "Profit",
              hline = elementLine(linetype = "solid", size = 1),
              connector = elementLine(linetype = "dotted"),
              label = elementText(size = 20, family = "Courier", face = "bold"),
              labelFormat = "$~s") +
    scaleYContinuous(name = "Values", format = "$~s") +
    ggtitle("Company Profit (in USD)") +
    ggsize(1000, 500) +
    themeMinimal() +
    theme(plotTitle = elementText(size = 20, face = "bold", hjust = 0.5))

## Additional Parameters

### `measure` and `group`

In [5]:
val groupSize = 7
val dataWithGroupsMap = mapOf(
    "Company" to List(groupSize) { "Badgersoft" } + List(groupSize) { "AIlien Co." },
    "Accounts" to listOf("initial", "revenue", "costs", "Q1", "revenue", "costs", "Q2").let { it + it },
    "Values" to listOf(200, 200, -100, null, 250, -100, null,
                       150,  50, -100, null, 100, -100, null),
    "Measure" to listOf("absolute", "relative", "relative", "total", "relative", "relative", "total").let { it + it },
)
val companyMap = dataWithGroupsMap.entries.associate { (k, v) -> k to v.subList(0, groupSize) }

waterfallPlot(dataWithGroupsMap, "Accounts", "Values", measure = "Measure", group = "Company") +
    facetGrid(x = "Company", scales = "free_x")

### `calcTotal`

`calcTotal = false` disables the calculation of the total.

If the `measure` serie is specified however, the `calcTotal` setting has no effect.

In [6]:
gggrid(listOf(
    waterfallPlot(dataMap, "Accounts", "Values", calcTotal = false),
    waterfallPlot(companyMap, "Accounts", "Values", measure = "Measure", calcTotal = false),
))

### Labels

There are several parameters that allow you to control the text labels on the waterfalls:

- `relativeLabels`: content and formatting of annotation labels on relative change bars (result of the call to the `layerLabels()` function);

- `absoluteLabels`: content and formatting of annotation labels on absolute value bars (result of the call to the `layerLabels()` function);

- `label`: style settings for all text labels (result of the call to the `elementText()` function).

In [7]:
waterfallPlot(dataMap, "Accounts", "Values", relativeLabels = layerLabels().line("@{..flow_type..}d:\n@..label.."),
                                             absoluteLabels = layerLabels().line("Result:\n@..label.."),
                                             label = elementText(face = "bold_italic"))

#### Hiding Labels

In [8]:
gggrid(listOf(
    waterfallPlot(dataMap, "Accounts", "Values", relativeLabels = "none") + ggtitle("Hide relative labels only"),
    waterfallPlot(dataMap, "Accounts", "Values", absoluteLabels = "none") + ggtitle("Hide relative labels only"),
    waterfallPlot(dataMap, "Accounts", "Values", label = "blank") + ggtitle("Hide all labels"),
))

### Tooltips

Tooltips for relative and absolute measures should be specified independently.

In [9]:
val relativeTooltips = layerTooltips()
    .title("Account: @..xlabel..")
    .format("@..initial..", " $,.2~s")
    .format("@..value..", " $,.2~s")
    .line("@{..flow_type..}d from @..initial.. to @..value..")
    .disableSplitting()

gggrid(listOf(
    waterfallPlot(dataMap, "Accounts", "Values", relativeTooltips = "detailed", absoluteTooltips = "detailed") + ggtitle("'detailed' tooltips"),
    waterfallPlot(dataMap, "Accounts", "Values", relativeTooltips = relativeTooltips, absoluteTooltips = "none") + ggtitle("Custom tooltips"),
))

### `sortedValue`

In [10]:
waterfallPlot(dataMap, "Accounts", "Values", sortedValue = true)

### `threshold`/`maxValues`

In [11]:
gggrid(listOf(
    waterfallPlot(dataMap, "Accounts", "Values") + ggtitle("Default"),
    waterfallPlot(dataMap, "Accounts", "Values", threshold = 300_000) + ggtitle("Specified threshold"),
    waterfallPlot(dataMap, "Accounts", "Values", maxValues = 2) + ggtitle("Specified maxValues"),
))

### `base`

In [12]:
waterfallPlot(dataMap, "Accounts", "Values", base = 400_000)

### Combining `waterfallPlot()` with Other Geometry Layers

Waterfall plots can be enhanced by adding background and foreground layers. Foreground layers can be added using the regular `+` operator. Background layers can be added using the `backgroundLayers` parameter.

Limitations:

- layers must provide their own data;
- data coordinates must be numeric.

In [13]:
// background layer and its data
val quarterMap = mapOf(
    "period_start" to listOf(0.5, 3.5),
    "period_end" to listOf(3.5, 6.5),
    "ai_introduced" to listOf(false, true),
)
val quarterLayer = geomBand(
    data = quarterMap,
    alpha = 0.2,
    // we use "paint_a" to color the bands based on a separate category (e.g., quarters),
    // so they have their own color palette independent from the waterfalls
    fillBy = "paint_a", colorBy = "paint_a"
) {
    xmin="period_start";
    xmax="period_end";
    paint_a="ai_introduced"
}

// foreground layers and their data
val quarterLabelMap = mapOf(
    "name" to listOf("Q1", "Q2"),
    "x" to listOf(2, 5),
    "y" to listOf(600, 600),
)
val quarterAIStatusMap = mapOf(
    "text" to listOf("Before AI\nintroduction", "After AI\nintroduction"),
    "x" to listOf(1.5, 4.5),
    "y" to listOf(100, 100),
)
val textLayers = geomText(data = quarterLabelMap, size = 8) { x = "x"; y = "y"; label = "name" } +
    geomText(data=quarterAIStatusMap, size = 12) { x = "x"; y = "y"; label = "text" }

// whole plot
(waterfallPlot(companyMap, "Accounts", "Values", measure = "Measure",
                backgroundLayers = quarterLayer)  // background layer
  + textLayers                                    // foreground layers
  + scaleHue("paint_a", guide = "none")           // color for the background layer (bands)
  + ggtitle("Waterfall with additional layers"))

## Customize Colors

Let's look at the names of the flow types using the `showLegend` parameter:

In [14]:
val wp = waterfallPlot(companyMap, "Accounts", "Values", measure = "Measure", showLegend = true)
wp

Use these names to customize the colors:

In [15]:
wp + scaleFillManual(values = mapOf(
    "Increase" to "#66c2a5",
    "Decrease" to "#fc8d62",
    "Absolute" to "#e78ac3",
    "Total" to "#8da0cb",
))

If desired, you can also change the names of the flow types in the legend:

In [16]:
wp + scaleFillManual(values = mapOf(
    "Increase" to "#66c2a5",
    "Decrease" to "#fc8d62",
    "Absolute" to "#e78ac3",
    "Total" to "#8da0cb",
), labels = listOf("inc", "dec", "abs", "total"))

You can use a constant color for boxes and `"flow_type"` color for their borders:

In [17]:
waterfallPlot(companyMap, "Accounts", "Values", measure = "Measure", size = .75,
              fill = "gray90", color = "flow_type")

To paint the text labels, combine `color = "flow_type"` and `label = elementText(color = "inherit")`:

In [18]:
waterfallPlot(companyMap, "Accounts", "Values", measure = "Measure",
              fill = "gray90",
              color = "flow_type",                    //  Needed for mapping color to flow type
              label = elementText(color = "inherit")) //  Needed to inherit the text label color from the color of boxes border

The same can be done, for example, only for the relative text labels:

In [19]:
waterfallPlot(companyMap, "Accounts", "Values", measure = "Measure",
              fill = "gray90",
              color = "flow_type",                                              //  Map color to flow type
              label = elementText(color = "indigo"),                            //  Choose some default color for absolute labels
              relativeLabels = layerLabels().line("@..label..").inheritColor()) //  Inherit color for the relative text labels